# 03 · O que muda no DuckDB 2.0  ⏱ 20 min

**DuckDB 2.0.0 LTS** saiu em **21/out/2026** (suporte até pelo menos out/2027). Quase 20 mil commits desde a 1.5. Segundo o próprio anúncio, é "acima de tudo uma versão de recursos", com poucas quebras de compatibilidade.

| Tema | O que chegou |
|---|---|
| Arquitetura | Servidor/cliente **Quack** e `CONNECT`, I/O assíncrono |
| SQL | **Triggers**, CTEs que modificam dados, `$variavel`, schemas aninhados, `FETCH FIRST`, funções de mutação de JSON |
| Tipos | **VARIANT** de primeira classe (com *shredding* em Parquet) |
| Ferramentas | Novo parser PEG (padrão), CLI com **modo agente**, repositórios de extensões |
| Lakehouse | DuckLake 1.1, Iceberg e Delta melhores |

As quebras de compatibilidade afetam o formato de arquivo, as lambdas e a API Python. A seção 1 trata das três.

In [ ]:
import duckdb, os, subprocess, shutil
print(duckdb.__version__)
con = duckdb.connect()

def cli(sql, *flags):
    """Roda o CLI do DuckDB (pip install duckdb-cli) e mostra a saída."""
    if not shutil.which("duckdb"):
        print("CLI não encontrado: pip install duckdb-cli")
        return
    r = subprocess.run(["duckdb", *flags, "-c", sql], capture_output=True, text=True,
                       stdin=subprocess.DEVNULL)
    print(r.stdout or r.stderr)

## 1. Quebras de compatibilidade (comece por aqui)

### 1.1 Formato de arquivo
Um `.duckdb` gravado pela 2.0 **não abre** na 1.x. Se outra equipe ainda usa 1.4, grave com a versão de armazenamento antiga:

In [ ]:
if os.path.exists("compat_1_4.duckdb"):
    os.remove("compat_1_4.duckdb")
con.sql("ATTACH 'compat_1_4.duckdb' AS compat (STORAGE_VERSION 'v1.4.0')")
con.sql("CREATE TABLE compat.exemplo AS SELECT 1 AS id")
con.sql("SELECT database_name, path, options FROM duckdb_databases() WHERE database_name = 'compat'").show()
_ = con.sql("DETACH compat")

### 1.2 Lambdas: `x -> x + 1` foi desligado
A seta única conflitava com o operador JSON `->`. Use a palavra-chave `lambda`. (É possível religar com `SET lambda_syntax = 'enable_single_arrow'`, mas isso some na 2.1.)

In [ ]:
try:
    con.sql("SELECT list_transform([1, 2, 3], x -> x * 2)").show()
except Exception as e:
    print("Erro:", str(e).splitlines()[0])

con.sql("SELECT list_transform([1, 2, 3], lambda x: x * 2) AS dobro").show()

### 1.3 API Python mais rígida
Na 2.0 o cliente Python foi reescrito. Duas mudanças costumam quebrar scripts antigos:

- `relação.fetchone()` não existe mais (a relação é re-executada a cada chamada). Use `.fetchall()`, `.df()`, `.pl()`.
- Um `con.execute(...)` deixa o resultado "aberto"; consuma tudo (`.fetchall()`) antes de usar a conexão de novo, ou use `con.cursor()` para resultados paralelos.

In [ ]:
rel = con.sql("SELECT 42 AS resposta")
try:
    rel.fetchone()
except AttributeError as e:
    print("AttributeError:", str(e)[:90], "…")
print(rel.fetchall())

### 1.4 Novo parser (PEG) com mensagens melhores
O parser novo é o padrão e aponta exatamente onde está o erro. Se uma query antiga quebrar, volte ao parser anterior com `SET enable_legacy_parser = true`.

In [ ]:
try:
    con.sql("SELECT nome, cidade FROMM clientes")
except Exception as e:
    print(e)

## 2. Triggers

A 2.0 suporta `BEFORE`/`AFTER`, `FOR EACH ROW`/`FOR EACH STATEMENT` e tabelas de transição (`REFERENCING OLD TABLE AS … NEW TABLE AS …`).

O exemplo abaixo grava numa tabela de auditoria cada mudança de preço.

In [ ]:
con.sql("""
    CREATE OR REPLACE TABLE produtos AS
    SELECT produto_id, nome, preco FROM 'dados/produtos.json' WHERE produto_id <= 5;

    CREATE OR REPLACE TABLE auditoria_preco (
        produto_id INTEGER, preco_antigo DOUBLE, preco_novo DOUBLE, alterado_em TIMESTAMP
    );

    CREATE TRIGGER trg_auditoria_preco
    AFTER UPDATE ON produtos
    REFERENCING OLD TABLE AS antigo NEW TABLE AS novo
    FOR EACH STATEMENT
    INSERT INTO auditoria_preco
        SELECT antigo.produto_id, antigo.preco, novo.preco, now()
        FROM antigo JOIN novo USING (produto_id)
        WHERE antigo.preco <> novo.preco;
""")

con.sql("UPDATE produtos SET preco = round(preco * 1.10, 2) WHERE produto_id IN (1, 2)")
con.sql("FROM auditoria_preco").show()

> Algumas combinações (por exemplo `BEFORE … FOR EACH ROW` com `UPDATE`) ainda não existem, e `MERGE INTO` não roda em tabela que tem trigger.

In [ ]:
con.sql("FROM duckdb_triggers()").df()  # catálogo de triggers

In [ ]:
_ = con.sql("DROP TRIGGER trg_auditoria_preco ON produtos")

## 3. CTEs que modificam dados

`INSERT`, `UPDATE`, `DELETE` e `COPY` podem ser etapas de um pipeline dentro do `WITH`. O exemplo move vendas canceladas para uma tabela de arquivo, numa única instrução.

In [ ]:
con.sql("""
    CREATE OR REPLACE TABLE vendas_jan AS FROM 'dados/vendas_2025_01.csv';
    CREATE OR REPLACE TABLE vendas_canceladas AS FROM vendas_jan LIMIT 0;
""")

con.sql("""
    WITH removidas AS (
        DELETE FROM vendas_jan WHERE status = 'cancelado' RETURNING *
    ),
    arquivadas AS (
        INSERT INTO vendas_canceladas SELECT * FROM removidas RETURNING venda_id
    )
    SELECT count(*) AS movidas FROM arquivadas
""").show()

con.sql("SELECT (SELECT count(*) FROM vendas_jan) AS ativas, (SELECT count(*) FROM vendas_canceladas) AS canceladas").show()

## 4. Variáveis com `$`

`SET VARIABLE` já existia; agora `$nome` substitui `getvariable('nome')` **em SQL puro** (CLI, UI, scripts `.sql`).

> No Python, `$nome` é interpretado como parâmetro de consulta preparada, então lá continue usando `getvariable()` ou parâmetros.

In [ ]:
con.sql("SET VARIABLE meta_receita = 700")
con.sql("SELECT getvariable('meta_receita') * 1.1 AS meta_com_folga").show()

In [ ]:
# o mesmo no CLI, com a sintaxe nova
cli("SET VARIABLE meta_receita = 700; SELECT $meta_receita * 1.1 AS meta_com_folga;", "-no-agent")

## 5. VARIANT: JSON com desempenho de coluna

`VARIANT` guarda valores de tipos diferentes na mesma coluna, mas com tipo binário (não texto). Na 2.0 ganhou *shredding*: campos frequentes viram colunas internas, e o Parquet lê e grava isso.

In [ ]:
con.sql("""
    CREATE OR REPLACE TABLE eventos AS
    SELECT * FROM (VALUES
        (1, {'tipo': 'clique', 'botao': 'comprar'}::VARIANT),
        (2, {'tipo': 'compra', 'valor': 199.9, 'itens': [10, 20]}::VARIANT),
        (3, 'texto livre'::VARIANT),
        (4, 42::VARIANT)
    ) t(id, payload)
""")
con.sql("""
    SELECT id,
           variant_typeof(payload)  AS tipo,
           variant_keys(payload)    AS chaves,
           payload.tipo             AS evento,
           payload.valor::DOUBLE    AS valor
    FROM eventos
""").show()

In [ ]:
# JSON -> VARIANT, e VARIANT em Parquet
con.sql("""
    COPY (
        SELECT ts, contexto::VARIANT AS contexto
        FROM read_json('dados/logs/app_2025-03-01.jsonl')
    ) TO 'dados/logs_variant.parquet'
""")
con.sql("DESCRIBE FROM 'dados/logs_variant.parquet'").show()
con.sql("""
    SELECT contexto.versao_app::VARCHAR AS versao, count(*) AS n
    FROM 'dados/logs_variant.parquet'
    GROUP BY ALL ORDER BY n DESC
""").show()

## 6. Funções de mutação de JSON

`json_set`, `json_insert`, `json_replace`, `json_remove`: editar JSON sem converter para struct e voltar.

In [ ]:
con.sql("""
    SELECT
        json_set('{"nome": "Ana"}', '$.cidade', '"Recife"'::JSON)          AS com_set,
        json_remove('{"nome": "Ana", "cpf": "000"}', '$.cpf')               AS sem_cpf
""").show()

## 7. SQL padrão e organização

- `FETCH FIRST n ROWS ONLY` (padrão SQL, alternativa ao `LIMIT`)
- `UNNEST` dentro do `GROUP BY`
- Schemas aninhados: `CREATE SCHEMA financeiro.relatorios`

In [ ]:
con.sql("FROM range(100) FETCH FIRST 3 ROWS ONLY").show()

con.sql("""
    SELECT unnest(['a', 'b', 'a', 'c', 'a']) AS letra, count(*) AS n
    GROUP BY unnest(['a', 'b', 'a', 'c', 'a'])
    ORDER BY n DESC
""").show()

con.sql("CREATE SCHEMA IF NOT EXISTS financeiro; CREATE SCHEMA IF NOT EXISTS financeiro.relatorios")
con.sql("CREATE OR REPLACE TABLE financeiro.relatorios.metas AS SELECT 'site' AS canal, 700 AS meta")
con.sql("FROM financeiro.relatorios.metas").show()

## 8. Quack: DuckDB como servidor

Até a 1.x, só um processo podia escrever num arquivo `.duckdb`. A 2.0 traz um servidor/cliente nativo (**Quack**) e o comando `CONNECT`, que também funciona com Postgres e MySQL e empurra a consulta para o servidor remoto.

**Demonstração ao vivo em dois terminais** (precisa de internet na primeira vez para baixar a extensão):

```sql
-- terminal 1: servidor
duckdb curso.duckdb
CALL quack_serve(token = 'aula');

-- terminal 2: cliente
duckdb
ATTACH 'quack:localhost' AS remoto (TOKEN 'aula');
CONNECT remoto;
SELECT count(*) FROM vendas;
```

Atalhos no CLI: `duckdb -serve` e `duckdb -connect`. Por padrão escuta só em `localhost`.

## 9. CLI com modo agente

Quando um agente de IA (Claude Code, Cursor…) roda o `duckdb`, a CLI detecta e muda a saída: tabelas Markdown compactas, erros em JSON, limites de linhas. Segundo o anúncio, reduziu em 59% o texto gerado no TPC-H. Força com `-agent`, desliga com `-no-agent`.

In [ ]:
cli("SELECT canal, count(*) AS n FROM 'dados/vendas_2025_01.csv' GROUP BY ALL ORDER BY ALL", "-agent")
cli("SELEC 1", "-agent")

## Checklist de migração da 1.x para a 2.0

1. Troque `x -> …` por `lambda x: …` nas suas queries.
2. Se outras ferramentas leem seus `.duckdb`, use `STORAGE_VERSION 'v1.4.0'` até elas atualizarem.
3. No Python, troque `rel.fetchone()` por `rel.fetchall()` e consuma resultados de `execute()`.
4. Rode seus testes, porque o parser novo é mais rígido em alguns casos (`enable_legacy_parser` é o plano B).
5. Leia o guia oficial de migração em <https://duckdb.org/docs/2.0/operations_manual/migration>.

In [ ]:
con.close()